# Exploratory Data Analysis — JUMP Cell Painting (Mito Channel)

**Thesis:** Unsupervised Phenotype Discovery in Cell Painting  
**Researcher:** Stephan Dsoud  
**Date:** March 2026  

This notebook performs a systematic EDA of the JUMP Cell Painting dataset before any modelling.
The goal is to understand what we have, how it is structured, and whether the data meets the
assumptions of our intended methods.

---

## Table of Contents

1. [Setup & Imports](#1)
2. [Data Audit](#2) — file counts, sizes, completeness
3. [Metadata Inspection](#3) — per-plate structure, role balance, compound coverage
4. [Glossary of Terms](#4) — biological and technical terminology
5. [Corpus-Level Exploration](#5) — overall structure and summary statistics
6. [Variable-Level Exploration](#6) — univariate feature distributions
7. [Interaction-Level Exploration](#7) — correlations, pairwise relationships
8. [DINOv3 Feature Exploration](#8) — extracted embeddings
9. [Summary & Take-Aways](#9)

<a id='1'></a>
## 1. Setup & Imports

In [ ]:
import os
import re
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from scipy import stats as sp_stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.05)
%matplotlib inline

# Paths
ROOT       = Path('.')
DATA_DIR   = ROOT / 'data'
META_DIR   = ROOT / 'metadata'
FEAT_NPZ   = ROOT / 'results' / 'compound_analysis_dinov3_vitb16' / 'features_compound.npz'
CLUSTER_CSV = ROOT / 'results' / 'compound_analysis_dinov3_vitb16' / 'compound_cluster_mapping.csv'

print('Ready.')

<a id='2'></a>
## 2. Data Audit

Before looking at any numbers, we inventory **what files exist**, how large they are,
and whether anything is missing. This is the "count your boxes before opening them" step.

### 2.1 Image Data Inventory

Each plate lives in `data/<plate_id>-Mito/<plate_id>-Mito/` and contains 16-bit TIFF images
named `<assay>_<well>_s<site>_w5<hash>.tif`.

In [ ]:
# Count plate directories
plate_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.endswith('-Mito')])
print(f'Plate directories found: {len(plate_dirs)}')

# Count TIFFs per plate (sample first 10 for speed, then extrapolate)
tiff_counts = {}
for pd_ in plate_dirs:
    inner = pd_ / pd_.name
    if inner.exists():
        tiffs = list(inner.glob('*.tif'))
        tiff_counts[pd_.name] = len(tiffs)
    else:
        tiff_counts[pd_.name] = 0

tiff_series = pd.Series(tiff_counts)
total_tiffs = tiff_series.sum()

print(f'Total TIFF images:       {total_tiffs:,}')
print(f'Plates with 0 TIFFs:     {(tiff_series == 0).sum()}')
print(f'TIFFs per plate — min:   {tiff_series.min()}')
print(f'TIFFs per plate — max:   {tiff_series.max()}')
print(f'TIFFs per plate — median: {tiff_series.median():.0f}')

In [ ]:
# Check a single TIFF for image dimensions and dtype
import tifffile

sample_plate = plate_dirs[0]
sample_tiff = list((sample_plate / sample_plate.name).glob('*.tif'))[0]
img = tifffile.imread(str(sample_tiff))

print(f'Sample image: {sample_tiff.name}')
print(f'  Shape: {img.shape}')
print(f'  Dtype: {img.dtype}')
print(f'  Value range: [{img.min()}, {img.max()}]')
print(f'  File size: {sample_tiff.stat().st_size / 1024:.1f} KB')

### 2.2 Metadata File Inventory

In [ ]:
meta_files = sorted(META_DIR.glob('meta*.csv'))
print(f'Metadata CSV files: {len(meta_files)}')

# Check alignment: every plate dir should have a metadata file and vice versa
plate_ids_from_dirs = {d.name.replace('-Mito', '') for d in plate_dirs}
plate_ids_from_meta = {f.stem.replace('meta', '') for f in meta_files}

dirs_no_meta = plate_ids_from_dirs - plate_ids_from_meta
meta_no_dirs = plate_ids_from_meta - plate_ids_from_dirs

print(f'\nAlignment check:')
print(f'  Plate dirs without metadata: {len(dirs_no_meta)} {list(dirs_no_meta)[:5] if dirs_no_meta else "(none)"}')
print(f'  Metadata without plate dir:  {len(meta_no_dirs)} {list(meta_no_dirs)[:5] if meta_no_dirs else "(none)"}')
print(f'  Matched plates:              {len(plate_ids_from_dirs & plate_ids_from_meta)}')

In [ ]:
# Metadata file sizes
meta_sizes = pd.Series({f.stem: f.stat().st_size / (1024**2) for f in meta_files})

print(f'Metadata file sizes (MB):')
print(f'  min:    {meta_sizes.min():.2f}')
print(f'  max:    {meta_sizes.max():.2f}')
print(f'  mean:   {meta_sizes.mean():.2f}')
print(f'  total:  {meta_sizes.sum():.1f} MB')

### 2.3 Datafiles Overview Table

A single reference table summarising every data source used in this project.

In [ ]:
overview = pd.DataFrame([
    {'Source': 'TIFF images', 'Location': 'data/<plate>-Mito/', 'Count': f'{total_tiffs:,} files across {len(plate_dirs)} plates',
     'Format': '16-bit grayscale TIFF', 'Typical Size': f'{sample_tiff.stat().st_size/1024:.0f} KB/image'},
    {'Source': 'Metadata CSVs', 'Location': 'metadata/meta<plate>.csv', 'Count': f'{len(meta_files)} files',
     'Format': 'CSV (1,800 columns)', 'Typical Size': f'{meta_sizes.mean():.1f} MB/plate'},
    {'Source': 'DINOv3 features', 'Location': 'results/.../features_compound.npz', 'Count': '1 file',
     'Format': 'NumPy NPZ', 'Typical Size': f'{FEAT_NPZ.stat().st_size/(1024**2):.1f} MB'},
    {'Source': 'Cluster mapping', 'Location': 'results/.../compound_cluster_mapping.csv', 'Count': '1 file',
     'Format': 'CSV (6 columns)', 'Typical Size': f'{CLUSTER_CSV.stat().st_size/1024:.0f} KB'},
])

overview.style.set_properties(**{'text-align': 'left'}).set_caption('Table 1: Datafiles Overview')

### 2.4 DINOv3 Feature File Audit

In [ ]:
npz = np.load(FEAT_NPZ, allow_pickle=True)
print('Keys in features_compound.npz:')
for k in npz.keys():
    v = npz[k]
    print(f'  {k:40s} shape={str(v.shape):20s} dtype={v.dtype}')

compound_ids  = npz['compound_ids']
X_raw         = npz['compound_features'].astype(np.float32)
X_clust       = npz['compound_features_for_clustering'].astype(np.float32)
umap_emb      = npz['umap_embedding'].astype(np.float32)
clusters      = npz['clusters']
feat_mask     = npz['well_feature_mask']

cluster_df = pd.read_csv(CLUSTER_CSV)

print(f'\nCompounds: {len(compound_ids):,}')
print(f'Raw feature dims: {X_raw.shape[1]}')
print(f'Selected feature dims: {X_clust.shape[1]}')
print(f'UMAP embedding: {umap_emb.shape}')
print(f'Clusters: {len(np.unique(clusters))} unique')
print(f'Feature mask: {feat_mask.sum()}/{len(feat_mask)} dims kept')

<a id='3'></a>
## 3. Metadata Inspection

Each plate's CSV contains **17 metadata columns** and **1,783 CellProfiler morphological features**
measured per well. We load all metadata (essential columns only) to understand the experimental design.

In [ ]:
essential_cols = [
    'Metadata_Plate', 'Metadata_Well', 'Metadata_ASSAY_WELL_ROLE',
    'Metadata_broad_sample', 'Metadata_pert_id', 'Metadata_pert_type',
    'Metadata_mmoles_per_liter', 'Metadata_broad_sample_type',
]

frames = []
for f in meta_files:
    try:
        header = pd.read_csv(f, nrows=0).columns
        usecols = [c for c in essential_cols if c in header]
        df = pd.read_csv(f, usecols=usecols)
        df['plate_id'] = f.stem.replace('meta', '')
        frames.append(df)
    except Exception as e:
        print(f'  WARN: {f.name}: {e}')

meta = pd.concat(frames, ignore_index=True)
meta['role'] = meta.get('Metadata_ASSAY_WELL_ROLE', 'unknown').astype(str).str.lower()
print(f'Loaded {len(meta):,} wells across {meta["plate_id"].nunique()} plates')

### 3.1 Metadata Column Overview

In [ ]:
# Show all metadata columns from one plate with dtypes, nulls, and sample values
sample_meta = pd.read_csv(meta_files[0], nrows=100)
meta_cols = [c for c in sample_meta.columns if c.startswith('Metadata_')]

col_info = pd.DataFrame({
    'Column': meta_cols,
    'Dtype': [str(sample_meta[c].dtype) for c in meta_cols],
    'Non-Null (%)': [(sample_meta[c].notna().mean() * 100) for c in meta_cols],
    'Unique Values': [sample_meta[c].nunique() for c in meta_cols],
    'Example': [str(sample_meta[c].dropna().iloc[0])[:40] if sample_meta[c].notna().any() else 'N/A' for c in meta_cols],
}).round(1)

col_info.style.set_caption('Table 2: Metadata Column Summary (Plate 24277)')

### 3.2 CellProfiler Feature Compartment Breakdown

In [ ]:
all_cols = sample_meta.columns.tolist()
cells_cols = [c for c in all_cols if c.startswith('Cells_')]
nuclei_cols = [c for c in all_cols if c.startswith('Nuclei_')]
cyto_cols = [c for c in all_cols if c.startswith('Cytoplasm_')]

# Sub-categories within each compartment
def count_subcategories(cols):
    cats = Counter()
    for c in cols:
        parts = c.split('_')
        if len(parts) >= 3:
            cats[parts[1]] += 1
    return dict(cats)

compartment_summary = pd.DataFrame([
    {'Compartment': 'Cells', 'Total Features': len(cells_cols),
     'Sub-categories': ', '.join(f'{k}({v})' for k, v in sorted(count_subcategories(cells_cols).items()))},
    {'Compartment': 'Nuclei', 'Total Features': len(nuclei_cols),
     'Sub-categories': ', '.join(f'{k}({v})' for k, v in sorted(count_subcategories(nuclei_cols).items()))},
    {'Compartment': 'Cytoplasm', 'Total Features': len(cyto_cols),
     'Sub-categories': ', '.join(f'{k}({v})' for k, v in sorted(count_subcategories(cyto_cols).items()))},
])

print(f'Total columns: {len(all_cols)}')
print(f'  Metadata:   {len(meta_cols)}')
print(f'  Cells:      {len(cells_cols)}')
print(f'  Nuclei:     {len(nuclei_cols)}')
print(f'  Cytoplasm:  {len(cyto_cols)}')
print()
compartment_summary.style.set_caption('Table 3: CellProfiler Feature Compartments')

### 3.3 Well Roles and Perturbation Types

In [ ]:
print('=== Well Role Distribution ===')
role_counts = meta['role'].value_counts()
for role, n in role_counts.items():
    print(f'  {role:15s} {n:>8,} wells ({n/len(meta)*100:5.1f}%)')

print(f'\n=== Perturbation Types ===')
if 'Metadata_pert_type' in meta.columns:
    pert_counts = meta['Metadata_pert_type'].value_counts()
    for pt, n in pert_counts.items():
        print(f'  {str(pt):15s} {n:>8,}')

print(f'\n=== Broad Sample Types ===')
if 'Metadata_broad_sample_type' in meta.columns:
    bst = meta['Metadata_broad_sample_type'].value_counts()
    for t, n in bst.items():
        print(f'  {str(t):15s} {n:>8,}')

### 3.4 Wells per Plate

In [ ]:
wells_per_plate = meta.groupby('plate_id').size()

print(f'Wells per plate:')
print(f'  min:    {wells_per_plate.min()}')
print(f'  max:    {wells_per_plate.max()}')
print(f'  mean:   {wells_per_plate.mean():.1f}')
print(f'  median: {wells_per_plate.median():.0f}')
print(f'  std:    {wells_per_plate.std():.1f}')

# Flag plates that differ significantly from the median
outlier_plates = wells_per_plate[abs(wells_per_plate - wells_per_plate.median()) > 2 * wells_per_plate.std()]
if len(outlier_plates) > 0:
    print(f'\n  Outlier plates ({len(outlier_plates)}):')
    for pid, n in outlier_plates.items():
        print(f'    {pid}: {n} wells')
else:
    print('\n  No outlier plates detected.')

### 3.5 Compound Coverage

In [ ]:
# How many unique compounds, how many wells per compound?
print('=== Compound Coverage (from cluster mapping) ===')
print(f'Unique compounds: {len(cluster_df):,}')
print(f'Roles: {cluster_df["role"].value_counts().to_dict()}')
print()
print(cluster_df[['n_wells', 'n_plates', 'concentration']].describe().round(2).to_string())

<a id='4'></a>
## 4. Glossary of Biological and Technical Terms

For readers unfamiliar with Cell Painting or the methods used, here is a reference table.

In [ ]:
glossary = pd.DataFrame([
    {'Term': 'Cell Painting', 'Category': 'Biology',
     'Definition': 'A high-content imaging assay using 6 fluorescent dyes to stain 8 cellular compartments, capturing morphological responses to perturbations.'},
    {'Term': 'JUMP Consortium', 'Category': 'Biology',
     'Definition': 'Joint Undertaking in Morphological Profiling — a large-scale collaborative producing standardised Cell Painting data for >100K compounds.'},
    {'Term': 'Mito Channel', 'Category': 'Biology',
     'Definition': 'The fluorescence channel capturing mitochondrial morphology (MitoTracker dye). Used as a single-channel proof-of-concept in this thesis.'},
    {'Term': 'DMSO', 'Category': 'Biology',
     'Definition': 'Dimethyl sulfoxide — the vehicle/solvent used to dissolve compounds. DMSO-only wells serve as negative controls (no perturbation).'},
    {'Term': 'Perturbation (pert)', 'Category': 'Biology',
     'Definition': 'A chemical or genetic treatment applied to cells. Here, small-molecule compounds identified by BRD (Broad) IDs.'},
    {'Term': 'Well / Microplate', 'Category': 'Biology',
     'Definition': 'A 384-well plate where each well contains cells treated with one compound at one concentration. Multiple sites imaged per well.'},
    {'Term': 'CellProfiler', 'Category': 'Technical',
     'Definition': 'Open-source software for quantitative image analysis. Extracts ~1,800 morphological features per well (AreaShape, Intensity, Texture, etc.).'},
    {'Term': 'DINOv3 (ViT-B/16)', 'Category': 'Technical',
     'Definition': 'A self-supervised Vision Transformer pre-trained on 1.7B images. Produces 768-dim feature vectors from the [CLS] token, capturing visual structure without task-specific labels.'},
    {'Term': 'Virtual Tiling', 'Category': 'Technical',
     'Definition': 'Splitting each image into a 3×4 grid of 12 tiles (each resized to 224×224) instead of downscaling. Preserves cellular-level detail.'},
    {'Term': 'Harmony', 'Category': 'Technical',
     'Definition': 'A batch correction algorithm using iterative soft k-means clustering to remove plate-level technical variation while preserving biological signal.'},
    {'Term': 'Leiden Algorithm', 'Category': 'Technical',
     'Definition': 'A community detection method on KNN graphs. Identifies clusters by maximising modularity. Used here on cosine-distance compound profiles.'},
    {'Term': 'UMAP', 'Category': 'Technical',
     'Definition': 'Uniform Manifold Approximation and Projection — a non-linear dimensionality reduction for 2D visualisation of high-dimensional data.'},
    {'Term': 'PCA', 'Category': 'Technical',
     'Definition': 'Principal Component Analysis — linear projection onto orthogonal axes of maximum variance. Used as intermediate step before clustering.'},
    {'Term': 'L2 Normalisation', 'Category': 'Technical',
     'Definition': 'Scaling each feature vector to unit length. After L2 norm, Euclidean distance equals cosine distance, simplifying KNN graph construction.'},
    {'Term': 'Fold Enrichment', 'Category': 'Statistics',
     'Definition': 'Ratio of observed hit fraction in a cluster to expected fraction by chance. A fold of 24× means hits are 24 times more concentrated than random.'},
    {'Term': 'iLISI', 'Category': 'Statistics',
     'Definition': 'Integration Local Inverse Simpson Index — measures how well batches are mixed locally. Higher values indicate better batch integration.'},
])

glossary.style.set_properties(**{'text-align': 'left'}).set_caption('Table 4: Glossary of Biological and Technical Terms')

<a id='5'></a>
## 5. Corpus-Level Exploration

We now look at the **overall shape** of the data: distributions of plate sizes,
role balance, cluster structure, and replicate coverage.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# (a) Wells per plate
ax = axes[0, 0]
ax.hist(wells_per_plate.values, bins=30, color='#264653', alpha=0.85, edgecolor='white')
ax.axvline(wells_per_plate.median(), color='#e76f51', ls='--', lw=2,
           label=f'Median: {wells_per_plate.median():.0f}')
ax.set_xlabel('Wells per plate'); ax.set_ylabel('Plates')
ax.set_title('(a) Wells per Plate'); ax.legend()

# (b) TIFFs per plate
ax = axes[0, 1]
ax.hist(tiff_series.values, bins=30, color='#2a9d8f', alpha=0.85, edgecolor='white')
ax.axvline(tiff_series.median(), color='#e76f51', ls='--', lw=2,
           label=f'Median: {tiff_series.median():.0f}')
ax.set_xlabel('TIFFs per plate'); ax.set_ylabel('Plates')
ax.set_title('(b) Images per Plate'); ax.legend()

# (c) Role balance
ax = axes[0, 2]
role_counts = meta['role'].value_counts()
colors_role = ['#3498db' if r == 'treated' else '#2ecc71' if r == 'mock' else '#95a5a6'
               for r in role_counts.index]
bars = ax.bar(role_counts.index, role_counts.values, color=colors_role, edgecolor='white')
for bar, val in zip(bars, role_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Wells'); ax.set_title('(c) Well Roles')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# (d) Cluster sizes
ax = axes[1, 0]
csizes = cluster_df['cluster'].value_counts().sort_index()
ax.bar(csizes.index, csizes.values, color='#f4a261', alpha=0.85, edgecolor='white')
ax.axhline(csizes.mean(), color='#e76f51', ls='--', label=f'Mean: {csizes.mean():.0f}')
ax.set_xlabel('Cluster'); ax.set_ylabel('Compounds')
ax.set_title(f'(d) Cluster Sizes (n={len(csizes)})'); ax.legend()

# (e) Replicate wells per compound
ax = axes[1, 1]
wpc = cluster_df['n_wells']
ax.hist(wpc, bins=range(1, wpc.max()+2), color='#e9c46a', alpha=0.85, edgecolor='white')
ax.axvline(wpc.median(), color='#264653', ls='--', lw=2,
           label=f'Median: {wpc.median():.0f}')
ax.set_xlabel('Wells per compound'); ax.set_ylabel('Compounds')
ax.set_title('(e) Replicate Wells'); ax.legend()

# (f) Concentration distribution
ax = axes[1, 2]
conc = cluster_df['concentration']
conc_nonzero = conc[conc > 0]
ax.hist(conc_nonzero, bins=50, color='#264653', alpha=0.7, edgecolor='none')
ax.set_xlabel('Concentration (mmol/L)'); ax.set_ylabel('Compounds')
ax.set_title(f'(f) Concentration Distribution (n={len(conc_nonzero):,})')

plt.suptitle('Corpus-Level Overview', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:**

- **(a)** Wells per plate cluster tightly around 384, confirming standard 384-well format. A few plates have slightly fewer wells (likely edge exclusions or QC failures).
- **(b)** TIFFs per plate (~2,304) = 384 wells × 6 sites/well. Consistent across plates — no missing imaging runs.
- **(c)** Treated wells outnumber mock ~5:1, reflecting the compound screening design where most wells receive a perturbation.
- **(d)** Leiden clustering produces 22 clusters of highly variable size. Cluster 0 (DMSO-dominated) is among the largest; small clusters may capture rare phenotypes.
- **(e)** The vast majority of compounds have exactly 4 replicate wells, enabling robust median aggregation.
- **(f)** Concentrations are concentrated around 5 and 10 µM — the two standard screening doses.

### 5.1 Role Balance per Plate

Is the treated:mock ratio consistent across all 406 plates, or are some plates unusual?

In [ ]:
role_by_plate = meta.groupby(['plate_id', 'role']).size().unstack(fill_value=0)
if 'mock' in role_by_plate.columns and 'treated' in role_by_plate.columns:
    role_by_plate['mock_pct'] = role_by_plate['mock'] / role_by_plate.sum(axis=1) * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.scatter(role_by_plate['treated'], role_by_plate['mock'], alpha=0.5, s=30, c='#264653')
    ax.set_xlabel('Treated wells'); ax.set_ylabel('Mock wells')
    ax.set_title('Treated vs Mock Wells per Plate')

    ax = axes[1]
    ax.hist(role_by_plate['mock_pct'], bins=30, color='#2ecc71', alpha=0.7, edgecolor='white')
    ax.set_xlabel('Mock wells (%)'); ax.set_ylabel('Plates')
    ax.set_title('Mock Fraction Distribution Across Plates')
    ax.axvline(role_by_plate['mock_pct'].median(), color='#264653', ls='--',
               label=f'Median: {role_by_plate["mock_pct"].median():.1f}%')
    ax.legend()

    plt.tight_layout()
    plt.show()
else:
    print('Only one role found — skipping.')

**Interpretation:** The mock fraction is remarkably consistent across plates (~17%), confirming
standardised plate layout. No plates deviate drastically, so batch correction (Harmony) operates
on a balanced experimental design.

<a id='6'></a>
## 6. Variable-Level Exploration

We look at **individual features** to understand their marginal distributions, detect
skewness and outliers, and compare treated vs mock wells.

### 6.1 Load CellProfiler Features (20-Plate Sample)

Loading all 406 plates with 1,800 columns would use too much memory. We sample 20 plates.

In [ ]:
rng = np.random.default_rng(42)
sample_files = rng.choice(meta_files, size=20, replace=False)

cp_frames = []
for f in sample_files:
    try:
        df = pd.read_csv(f)
        df['plate_id'] = f.stem.replace('meta', '')
        cp_frames.append(df)
    except Exception:
        pass

cp = pd.concat(cp_frames, ignore_index=True)
cp['role'] = cp.get('Metadata_ASSAY_WELL_ROLE', 'unknown').astype(str).str.lower()

feature_cols = [c for c in cp.columns if c.startswith(('Cells_', 'Nuclei_', 'Cytoplasm_'))]
cp_num = cp[feature_cols].apply(pd.to_numeric, errors='coerce')

print(f'Loaded {len(cp):,} wells from {len(sample_files)} plates')
print(f'Feature columns: {len(feature_cols)}')

### 6.2 Summary Statistics Table

In [ ]:
# Compute summary stats for all features
desc = cp_num.describe().T
desc['missing_pct'] = (cp_num.isna().mean() * 100).values
desc['skewness'] = cp_num.skew().values
desc['kurtosis'] = cp_num.kurtosis().values

print(f'=== Feature Summary Statistics (n={len(feature_cols)} features, {len(cp):,} wells) ===')
print(f'Features with >5% missing:  {(desc["missing_pct"] > 5).sum()}')
print(f'Features with |skew| > 2:   {(desc["skewness"].abs() > 2).sum()}')
print(f'Features with kurtosis > 10: {(desc["kurtosis"] > 10).sum()}')
print()

# Show the 10 most skewed features
print('Top 10 most skewed features:')
desc.nlargest(10, 'skewness')[['mean', 'std', 'min', 'max', 'skewness', 'kurtosis']].round(3)

### 6.3 Histograms + KDE — Representative CellProfiler Features

In [ ]:
# Pick 9 representative features across compartments and measurement types
candidates = [
    'Cells_AreaShape_Area', 'Cells_AreaShape_Eccentricity', 'Cells_AreaShape_FormFactor',
    'Nuclei_AreaShape_Area', 'Nuclei_Intensity_MeanIntensity_DNA', 'Nuclei_AreaShape_Compactness',
    'Cytoplasm_AreaShape_Area', 'Cytoplasm_Intensity_MeanIntensity_Mito', 'Cytoplasm_AreaShape_Eccentricity',
]
feats_to_plot = [c for c in candidates if c in cp_num.columns][:9]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flat, feats_to_plot):
    vals = cp_num[col].dropna()
    q01, q99 = vals.quantile(0.01), vals.quantile(0.99)
    vals_clip = vals[(vals >= q01) & (vals <= q99)]

    ax.hist(vals_clip, bins=60, density=True, color='#2a9d8f', alpha=0.5, edgecolor='none')
    try:
        vals_clip.plot.kde(ax=ax, color='#264653', linewidth=2)
    except Exception:
        pass

    short = col.replace('Cells_', 'C.').replace('Nuclei_', 'N.').replace('Cytoplasm_', 'Cy.')
    skew = vals.skew()
    ax.set_title(f'{short}\nskew={skew:.2f}', fontsize=9)
    ax.set_ylabel('Density', fontsize=8)
    ax.tick_params(labelsize=7)

# Turn off unused axes
for ax in axes.flat[len(feats_to_plot):]:
    ax.set_visible(False)

plt.suptitle('Univariate Distributions: Histograms + KDE (20-Plate Sample)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:** Area features (Cells, Nuclei, Cytoplasm) are **right-skewed**, which is typical for
cell size measurements — most cells are "normal" sized, with a tail of large cells. Shape descriptors
like Eccentricity and FormFactor are closer to symmetric. High skewness flags features that might
benefit from log-transformation if used in parametric models.

### 6.4 Boxplots — Treated vs Mock

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))

for ax, col in zip(axes.flat, feats_to_plot):
    data_plot = cp[[col, 'role']].copy()
    data_plot[col] = pd.to_numeric(data_plot[col], errors='coerce')
    data_plot = data_plot[data_plot['role'].isin(['treated', 'mock'])].dropna()
    q01, q99 = data_plot[col].quantile(0.01), data_plot[col].quantile(0.99)
    data_plot = data_plot[(data_plot[col] >= q01) & (data_plot[col] <= q99)]

    sns.boxplot(data=data_plot, x='role', y=col, ax=ax,
                palette={'treated': '#3498db', 'mock': '#2ecc71'},
                fliersize=1.5, linewidth=1)
    short = col.replace('Cells_', 'C.').replace('Nuclei_', 'N.').replace('Cytoplasm_', 'Cy.')
    ax.set_title(short, fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7)

for ax in axes.flat[len(feats_to_plot):]:
    ax.set_visible(False)

plt.suptitle('Boxplots by Well Role: Treated (blue) vs Mock/DMSO (green)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:** For most CellProfiler features, treated and mock distributions overlap
substantially. This is expected: most compounds have **weak or no effect**, and the few that
alter morphology are diluted by the majority. This motivates aggregation to the compound level
and high-dimensional clustering, rather than univariate thresholds.

### 6.5 Missing Values Heatmap

In [ ]:
# Show missing-value pattern across features (sampled for readability)
missing_pct = cp_num.isna().mean().sort_values(ascending=False)
has_missing = missing_pct[missing_pct > 0]

if len(has_missing) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(has_missing)), has_missing.values * 100, color='#e76f51', alpha=0.8)
    ax.set_xlabel(f'Features with missing values (n={len(has_missing)})')
    ax.set_ylabel('Missing (%)')
    ax.set_title('Missing Value Fraction Across CellProfiler Features')
    plt.tight_layout()
    plt.show()
    print(f'{len(has_missing)} features have at least one missing value.')
    print(f'Max missing: {missing_pct.iloc[0]*100:.2f}% in {missing_pct.index[0]}')
else:
    print('No missing values detected in the 20-plate sample.')

<a id='7'></a>
## 7. Interaction-Level Exploration

We now examine **relationships between features**: correlations within and across
compartments, and whether CellProfiler features form natural groupings.

### 7.1 Correlation Heatmap — CellProfiler (Sampled)

We sample 15 features from each compartment (45 total) for a readable heatmap.

In [ ]:
sample_cols = []
for prefix in ['Cells_', 'Nuclei_', 'Cytoplasm_']:
    available = [c for c in feature_cols if c.startswith(prefix)]
    chosen = [available[i] for i in np.linspace(0, len(available)-1, 15, dtype=int)]
    sample_cols.extend(chosen)

corr = cp_num[sample_cols].corr()

# Abbreviate labels
short_labels = [c.replace('Cells_', 'C.').replace('Nuclei_', 'N.').replace('Cytoplasm_', 'Cy.')
                .replace('AreaShape_', '').replace('Intensity_', 'I.').replace('Texture_', 'T.')[:28]
                for c in sample_cols]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, ax=ax, cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            xticklabels=short_labels, yticklabels=short_labels,
            linewidths=0.1, linecolor='#f0f0f0')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)

# Add compartment dividers
for pos in [15, 30]:
    ax.axhline(pos, color='black', linewidth=1.5)
    ax.axvline(pos, color='black', linewidth=1.5)

ax.set_title('CellProfiler Feature Correlations (45 sampled, grouped by compartment)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:** Strong within-compartment correlation blocks are visible — especially among AreaShape
and Intensity features. Cross-compartment correlations (off-diagonal blocks) are weaker but present,
reflecting genuine biological coupling (e.g., larger cells tend to have larger nuclei). The high redundancy
within compartments is exactly why correlation-based feature selection (|r| > 0.95) is applied.

### 7.2 Correlation Distribution — How Redundant Are the Features?

In [ ]:
# Compute full pairwise correlation on a feature subsample for speed
n_feat_sample = min(200, len(feature_cols))
feat_subsample = rng.choice(feature_cols, size=n_feat_sample, replace=False)
full_corr = cp_num[feat_subsample].corr()
upper_tri = full_corr.values[np.triu_indices_from(full_corr.values, k=1)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(upper_tri, bins=100, color='#264653', alpha=0.7, edgecolor='none')
axes[0].axvline(0.95, color='#e76f51', ls='--', lw=2, label='r = 0.95')
axes[0].axvline(-0.95, color='#e76f51', ls='--', lw=2)
n_high = int((np.abs(upper_tri) > 0.95).sum())
axes[0].set_xlabel('Pearson r')
axes[0].set_ylabel('Feature pairs')
axes[0].set_title(f'Pairwise Correlation Distribution\n({n_high:,} pairs with |r| > 0.95)')
axes[0].legend()

# What fraction is highly correlated?
thresholds = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]
fracs = [(np.abs(upper_tri) > t).mean() * 100 for t in thresholds]
axes[1].bar([str(t) for t in thresholds], fracs, color='#f4a261', alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Correlation threshold |r|')
axes[1].set_ylabel('% of feature pairs above threshold')
axes[1].set_title('Feature Redundancy at Various Thresholds')
for i, (t, f) in enumerate(zip(thresholds, fracs)):
    axes[1].text(i, f + 0.3, f'{f:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

### 7.3 Plate Effect — Feature Variance Across Plates

In [ ]:
# For 6 features, show the distribution across plates (violin or strip)
feats_for_plate = feats_to_plot[:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, feats_for_plate):
    data_plot = cp[[col, 'plate_id']].copy()
    data_plot[col] = pd.to_numeric(data_plot[col], errors='coerce')
    data_plot = data_plot.dropna()
    q01, q99 = data_plot[col].quantile(0.01), data_plot[col].quantile(0.99)
    data_plot = data_plot[(data_plot[col] >= q01) & (data_plot[col] <= q99)]

    plate_means = data_plot.groupby('plate_id')[col].mean().sort_values()
    ax.bar(range(len(plate_means)), plate_means.values, color='#2a9d8f', alpha=0.7)
    ax.axhline(plate_means.mean(), color='#e76f51', ls='--', lw=1.5)
    short = col.replace('Cells_', 'C.').replace('Nuclei_', 'N.').replace('Cytoplasm_', 'Cy.')
    ax.set_title(f'{short}\nCV={plate_means.std()/plate_means.mean()*100:.1f}%', fontsize=9)
    ax.set_xlabel('Plate (sorted)'); ax.set_ylabel('Mean value')
    ax.tick_params(labelsize=7)
    ax.set_xticks([])

plt.suptitle('Plate-to-Plate Variation in Feature Means (20 sampled plates)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:** Even within a 20-plate sample, plate-to-plate variation in feature means is visible
(CV values in titles). This is the **batch effect** that Harmony correction addresses. Features with
high inter-plate CV are most affected by technical variation.

<a id='8'></a>
## 8. DINOv3 Feature Exploration

The DINOv3 ViT-B/16 produces 768-dimensional features per tile, which are aggregated
to compound level (30,413 × 337 after feature selection). We explore these embeddings.

### 8.1 DINOv3 Feature Distributions

In [ ]:
from scipy.stats import gaussian_kde

dino_indices = np.linspace(0, X_clust.shape[1]-1, 9, dtype=int)

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, idx in zip(axes.flat, dino_indices):
    vals = X_clust[:, idx]
    ax.hist(vals, bins=60, density=True, color='#f4a261', alpha=0.5, edgecolor='none')
    try:
        kde = gaussian_kde(vals)
        x_grid = np.linspace(vals.min(), vals.max(), 200)
        ax.plot(x_grid, kde(x_grid), color='#264653', linewidth=2)
    except Exception:
        pass
    skew = sp_stats.skew(vals)
    kurt = sp_stats.kurtosis(vals)
    ax.set_title(f'Dim {idx} (skew={skew:.2f}, kurt={kurt:.2f})', fontsize=9)
    ax.set_ylabel('Density', fontsize=8)

plt.suptitle('DINOv3 Feature Distributions (Post-Selection, L2-Normalised)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:** Most DINOv3 dimensions are approximately unimodal and symmetric after L2 normalisation.
A few show mild bimodality, which likely corresponds to distinct phenotypic populations (e.g.,
DMSO-like vs active compounds).

### 8.2 PCA — Explained Variance

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_clust)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_90 = int(np.searchsorted(cumvar, 0.90)) + 1
n_95 = int(np.searchsorted(cumvar, 0.95)) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 51), pca.explained_variance_ratio_, color='#2a9d8f', alpha=0.7)
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Variance Ratio')
axes[0].set_title('Individual Explained Variance')

axes[1].plot(range(1, 51), cumvar, 'o-', color='#264653', markersize=3, linewidth=2)
axes[1].axhline(0.90, color='#e76f51', ls='--', label=f'90% at PC{n_90}')
axes[1].axhline(0.95, color='#e9c46a', ls='--', label=f'95% at PC{n_95}')
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Variance')
axes[1].set_title('Cumulative Explained Variance'); axes[1].legend()

plt.suptitle('PCA on DINOv3 Compound Features (337 dims)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance')
print(f'90% variance at PC{n_90}, 95% at PC{n_95}')

### 8.3 UMAP — Coloured by Cluster, Role, and Concentration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# (a) Clusters
n_cl = len(np.unique(clusters))
cmap = plt.cm.tab20 if n_cl <= 20 else plt.cm.nipy_spectral
colors = cmap(np.linspace(0, 1, n_cl))
for cid in range(n_cl):
    m = clusters == cid
    axes[0].scatter(umap_emb[m, 0], umap_emb[m, 1], s=2, alpha=0.4, c=[colors[cid]])
axes[0].set_title(f'(a) Leiden Clusters (n={n_cl})')
axes[0].set_xlabel('UMAP 1'); axes[0].set_ylabel('UMAP 2')

# (b) Role
for role, color in [('treated', '#3498db'), ('mock', '#2ecc71')]:
    m = cluster_df['role'].values == role
    if m.any():
        axes[1].scatter(umap_emb[m, 0], umap_emb[m, 1], s=2, alpha=0.3,
                        c=color, label=f'{role} ({m.sum():,})')
axes[1].set_title('(b) Compound Role')
axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2'); axes[1].legend(markerscale=5)

# (c) Concentration
conc_vals = cluster_df['concentration'].values
sc = axes[2].scatter(umap_emb[:, 0], umap_emb[:, 1], s=2, alpha=0.4,
                     c=conc_vals, cmap='viridis',
                     vmin=0, vmax=np.percentile(conc_vals[conc_vals > 0], 95))
plt.colorbar(sc, ax=axes[2], shrink=0.8, label='Concentration (µM)')
axes[2].set_title('(c) Concentration')
axes[2].set_xlabel('UMAP 1'); axes[2].set_ylabel('UMAP 2')

plt.suptitle('UMAP Embedding of 30,413 Compounds', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:**

- **(a)** Leiden identifies 22 clusters forming distinct regions in UMAP space. Large clusters represent common morphologies; small ones capture rare phenotypic responses.
- **(b)** Mock (DMSO) compounds are confined to one region, while treated compounds spread across the embedding — the pipeline successfully separates control from perturbation.
- **(c)** Concentration does not strongly correlate with UMAP position, suggesting that morphological phenotype (rather than dose) is the primary axis of variation at the compound level.

### 8.4 DINOv3 Feature Correlation Heatmap

In [ ]:
# Sample 50 dimensions evenly for a readable heatmap
dim_sample = np.linspace(0, X_clust.shape[1]-1, 50, dtype=int)
dino_corr = np.corrcoef(X_clust[:, dim_sample].T)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(dino_corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Pearson r')
ax.set_title('DINOv3 Feature Correlations (50 sampled dims, post-selection)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Feature index'); ax.set_ylabel('Feature index')
plt.tight_layout()
plt.show()

upper = dino_corr[np.triu_indices_from(dino_corr, k=1)]
print(f'DINOv3 pairwise |r| — mean: {np.abs(upper).mean():.3f}, max: {np.abs(upper).max():.3f}')
print(f'Pairs with |r| > 0.8: {(np.abs(upper) > 0.8).sum()}')

### 8.5 Cosine Similarity Matrix (Ordered by Cluster)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Sample ~500 compounds ordered by cluster for block-diagonal display
sampled = []
for cid in range(n_cl):
    idx = np.where(clusters == cid)[0]
    take = min(len(idx), max(2, 500 // n_cl))
    sampled.extend(rng.choice(idx, size=take, replace=False).tolist())
sampled = sorted(set(sampled))

Xs = X_clust[sampled]
cs = clusters[sampled]
order = np.argsort(cs)
Xs = Xs[order]
cs_ordered = cs[order]
sim = cosine_similarity(Xs)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim, cmap='RdBu_r', vmin=-0.3, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Cosine Similarity')

boundaries = np.where(np.diff(cs_ordered))[0]
for b in boundaries:
    ax.axhline(b+0.5, color='black', lw=0.5, alpha=0.5)
    ax.axvline(b+0.5, color='black', lw=0.5, alpha=0.5)

ax.set_title(f'Compound Cosine Similarity (n={len(sampled)}, ordered by cluster)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:** Clear block-diagonal structure confirms that Leiden clusters correspond to
genuinely similar feature profiles — not random partitions. The warm diagonal blocks show
high intra-cluster similarity, while off-diagonal cold regions indicate distinct phenotypes.

### 8.6 QQ Normality Check — Top PCs

Leiden clustering does not require normality, but PCA and Harmony benefit from approximately
symmetric inputs. We check the distributional shape of the top PCs.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i in range(8):
    ax = axes.flat[i]
    vals = X_pca[:, i]
    sp_stats.probplot(vals, dist='norm', plot=ax)
    ax.get_lines()[0].set(color='#2a9d8f', markersize=2, alpha=0.3)
    ax.get_lines()[1].set(color='#e76f51', linewidth=2)

    sub = rng.choice(vals, size=min(5000, len(vals)), replace=False)
    _, p = sp_stats.shapiro(sub)
    skew = sp_stats.skew(vals)
    ax.set_title(f'PC{i+1} (skew={skew:.2f}, Shapiro p={p:.1e})', fontsize=9)
    ax.set_xlabel(''); ax.set_ylabel('')

plt.suptitle('QQ Normality Plots — Top 8 PCs (DINOv3 Features)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:** Most PCs follow the normal reference closely in the central range, with
mild heavy-tailed departures. Shapiro p-values are low (expected for n=30K), but the visual
deviations are modest. This supports the use of cosine-based KNN + Leiden (non-parametric)
rather than Gaussian mixture models.

<a id='9'></a>
## 9. Summary & Take-Aways

| Aspect | Finding |
|--------|--------|
| **Data scale** | 407 plates, 153K wells, 30K compounds, ~940K TIFF images |
| **Completeness** | Metadata and images are well-aligned; minimal missing values |
| **Plate consistency** | Wells per plate tightly centred at 384; mock fraction ~17% across all plates |
| **Feature redundancy** | CellProfiler features are highly correlated within compartments; DINOv3 features (post-selection) are much less redundant |
| **Batch effects** | Plate-to-plate variation in CellProfiler means is visible — Harmony correction is justified |
| **Distributions** | Area features are right-skewed; DINOv3 features are approximately symmetric. Top PCs show mild heavy tails |
| **Cluster structure** | Cosine similarity matrix shows clear block-diagonal structure; 22 Leiden clusters are well-separated |
| **DMSO baseline** | DMSO occupies a single compact region in UMAP space, confirming it represents a stable control phenotype |
| **Concentration** | Does not dominate the embedding — phenotype varies more by compound identity than dose |

**Conclusion:** The data is clean, well-structured, and suitable for unsupervised phenotype discovery.
Batch effects exist but are addressable with Harmony. The high redundancy in CellProfiler features
motivates the use of self-supervised DINOv3 features, which provide a compact, less correlated
representation for downstream clustering.